# Experiments

In [22]:
import os
import subprocess
import sys

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.48",
         "polars", "fastexcel", "sentencepiece", "protobuf"],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

import torch
from sklearn.metrics import f1_score

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.twd.loader import load
from models.benchmarks import always_neutral, random_guess
from models.bow import bow
from models.frozen_probe import probe
from models.plm_finetune import finetune, predict
from models.rule_based import rule_model
from models.word2vec import word2vec
from results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
# flip to "chrono" to rerun everything on the chronological split
SPLIT = "benchmark"
CORPUS = "twd" if SPLIT == "benchmark" else "twd-chrono"
# chrono's partition is fixed, so the seed only varies weight init -- one run is enough.
# 78516 rather than 5768: 5768 collapsed roberta-large on the smaller chrono train set.
SEEDS = SHAH_SEEDS if SPLIT == "benchmark" else (78516,)
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


results -> G:\My Drive\thesis\results.csv | device: cpu


## Chance benchmarks

In [16]:
for MODEL, fn in [("random", random_guess), ("always-neutral", always_neutral)]:
    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
            print(f"{MODEL} seed {seed}: already done, skipping")
            continue

        train, test = load(SPLIT, seed=seed)
        pred = fn(train, test, seed=seed)
        true = test["label"].to_list()

        save_result(
            OUT,
            model=MODEL,
            corpus=CORPUS,
            seed=seed,
            epochs="",  # no training
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{MODEL} seed {seed}: saved")

random seed 5768: already done, skipping
random seed 78516: already done, skipping
random seed 944601: already done, skipping
always-neutral seed 5768: already done, skipping
always-neutral seed 78516: already done, skipping
always-neutral seed 944601: already done, skipping


## Rule-based

In [17]:
MODEL = "rule-based"

for seed in SEEDS:
    if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
        print(f"{MODEL} seed {seed}: already done, skipping")
        continue

    train, test = load(SPLIT, seed=seed)
    pred = rule_model(test["sentence"].to_list())
    true = test["label"].to_list()

    save_result(
        OUT,
        model=MODEL,
        corpus=CORPUS,
        seed=seed,
        epochs="",  # no training
        weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
        macro_f1=round(f1_score(true, pred, average="macro"), 4),
    )
    print(f"{MODEL} seed {seed}: saved")

rule-based seed 5768: already done, skipping
rule-based seed 78516: already done, skipping
rule-based seed 944601: already done, skipping


## Bag-of-words

In [18]:
MODEL = "bow"

for seed in SEEDS:
    if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
        print(f"{MODEL} seed {seed}: already done, skipping")
        continue

    train, test = load(SPLIT, seed=seed)
    # raw text, unigrams -- six preprocessing variants moved macro-F1 by <0.008
    pred = bow(train, test, seed=seed)
    true = test["label"].to_list()

    save_result(
        OUT,
        model=MODEL,
        corpus=CORPUS,
        seed=seed,
        epochs="",  # no training
        weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
        macro_f1=round(f1_score(true, pred, average="macro"), 4),
    )
    print(f"{MODEL} seed {seed}: saved")

bow seed 5768: already done, skipping
bow seed 78516: already done, skipping
bow seed 944601: already done, skipping


## Word2Vec

In [19]:
MODEL = "word2vec"

for seed in SEEDS:
    if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
        print(f"{MODEL} seed {seed}: already done, skipping")
        continue

    train, test = load(SPLIT, seed=seed)
    # first call downloads 1.6GB of GoogleNews vectors, then cached
    pred = word2vec(train, test, seed=seed)
    true = test["label"].to_list()

    save_result(
        OUT,
        model=MODEL,
        corpus=CORPUS,
        seed=seed,
        epochs="",  # no training
        weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
        macro_f1=round(f1_score(true, pred, average="macro"), 4),
    )
    print(f"{MODEL} seed {seed}: saved")

word2vec seed 5768: already done, skipping
word2vec seed 78516: already done, skipping
word2vec seed 944601: already done, skipping


## Frozen encoders

In [20]:
for name in SHAH_PLM:
    MODEL = f"frozen:{name}"
    cfg = SHAH_PLM[name]

    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
            print(f"{MODEL} seed {seed}: already done, skipping")
            continue

        train, test = load(SPLIT, seed=seed)
        pred = probe(
            train,
            test,
            model_name=cfg["model_name"],
            max_len=cfg.get("max_len", 256),
            device=DEVICE,
            seed=seed,
        )
        true = test["label"].to_list()

        save_result(
            OUT,
            model=MODEL,
            corpus=CORPUS,
            seed=seed,
            epochs="",  # no training
            weighted_f1=round(f1_score(true, pred, average="weighted"), 4),
            macro_f1=round(f1_score(true, pred, average="macro"), 4),
        )
        print(f"{MODEL} seed {seed}: saved")


frozen:bert-base-uncased seed 5768: already done, skipping
frozen:bert-base-uncased seed 78516: already done, skipping
frozen:bert-base-uncased seed 944601: already done, skipping
frozen:bert-large-uncased seed 5768: already done, skipping
frozen:bert-large-uncased seed 78516: already done, skipping
frozen:bert-large-uncased seed 944601: already done, skipping
frozen:roberta-base seed 5768: already done, skipping
frozen:roberta-base seed 78516: already done, skipping
frozen:roberta-base seed 944601: already done, skipping
frozen:roberta-large seed 5768: already done, skipping
frozen:roberta-large seed 78516: already done, skipping
frozen:roberta-large seed 944601: already done, skipping
frozen:flang-bert seed 5768: already done, skipping
frozen:flang-bert seed 78516: already done, skipping
frozen:flang-bert seed 944601: already done, skipping
frozen:flang-roberta seed 5768: already done, skipping
frozen:flang-roberta seed 78516: already done, skipping
frozen:flang-roberta seed 944601: 

## Fine-tuned encoders

In [ ]:
for MODEL in SHAH_PLM:
    cfg = SHAH_PLM[MODEL]

    for seed in SEEDS:
        if already_done(OUT, force=FORCE, model=MODEL, corpus=CORPUS, seed=seed):
            print(f"{MODEL} seed {seed}: already done, skipping")
            continue

        train, test = load(SPLIT, seed=seed)
        model, tok_, metrics = finetune(
            train,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            max_len=cfg.get("max_len", 256),
            seed=seed,
            test_df=test,
            device=DEVICE,
            verbose=True,
        )
        pred = predict(
            model, tok_, test, max_len=cfg.get("max_len", 256), device=DEVICE
        )
        mf1 = f1_score(test["label"].to_list(), pred, average="macro")

        save_result(
            OUT,
            model=MODEL,
            corpus=CORPUS,
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(mf1, 4),
        )
        print(f"{MODEL} seed {seed}: weighted={metrics['test_f1']:.4f}  macro={mf1:.4f}")

        # drop the model before the next seed builds one -- two large models plus
        # AdamW state don't fit on a 22GB L4
        del model, tok_
        torch.cuda.empty_cache()


Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


KeyboardInterrupt: 